Es muy común que durante el curso de una auditoría programada o una auditoría forense, se haga uso de información no estructurada: tal como correos electrónicos, pero sobre todo facturas como archivos .pdf. Seguramente recordarás que par esto siempre hay un auditor staff con menos experiencia, que tiene que hacer un vaciado factura por factura, de los principales datos a un archivo Excel... con el riesgo de manipulación y errores que ello lleva. Este notebook resume, de forma general, los pasos típicos para obtener la información a partir de unas facturas ficticias en .pdf.

In [1]:
# %pip install pdfplumber

In [2]:
import os
import re
import datetime
import pdfplumber
import pandas as pd

<p align="left">
  <img src="images/factura.png" alt="factura" width="400"><br>
  <em>ejemplo de la estructura de una factura ficticia </em>
</p>

In [3]:
# busca un archivo .pdf en la carpeta y verifica si lee información del pdf
pdf_folder = r'data\Facturas pdfs'
pdf_files = [f for f in os.listdir(pdf_folder) if f.endswith('.pdf')]
if pdf_files:
    pdf_path = os.path.join(pdf_folder, pdf_files[2])
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            print(f"\n--- Página {i + 1} ---")
            print(page.extract_text())
else:
    print("No se encontraron archivos PDF en la carpeta.")


--- Página 1 ---
Invoice: FX-5481
Global, Inc. Date: 02/02/2024
Client: LTD, Inc
SKU Ctd Description Cost Amount
LT01 24 Toy $ 21.30 $ 5 11.20
GTR21 64 The gun $ 24.50 $ 1,568.00
Total invoice amount: $ 2,079.20


In [ ]:
# --- configuración ---
pdf_folder = r'data\Facturas pdfs'
destino    = r'D:\tuUsuario\Carpeta\facturas_pdf.xlsx'
fecha_re   = re.compile(r'(\d{2}[/-]\d{2}[/-]\d{4})')
formatos   = ['%d-%m-%Y','%d/%m/%Y','%m-%d-%Y','%m/%d/%Y']

# función para obtener las fechas
def parse_fecha(s):
    for fmt in formatos:
        try: return datetime.datetime.strptime(s, fmt).strftime('%Y-%m-%d')
        except: pass
    return ''

# función para obtener la demás información
def extraer_info(texto):
    campos = {'Invoice':'','Client':'','Date':'','Total invoice amount':''}
    for l in (t:=texto.splitlines()):
        l=l.strip()
        if l.startswith('Invoice:'): campos['Invoice']=l.split(':',1)[1].strip()
        elif l.startswith('Client:'): campos['Client']=l.split(':',1)[1].strip()
        elif l.startswith('Total invoice amount:'): campos['Total invoice amount']=l.split(':',1)[1].strip()
        elif l.lower().startswith('date'):
            if (m:=fecha_re.search(l)): campos['Date']=parse_fecha(m.group(1))
    if not campos['Date'] and (m:=fecha_re.search(texto)):
        campos['Date']=parse_fecha(m.group(1))
    return campos

# --- procesamiento ---
filas=[]
for archivo in [f for f in os.listdir(pdf_folder) if f.lower().endswith('.pdf')]:
    with pdfplumber.open(os.path.join(pdf_folder, archivo)) as pdf:
        for i,pag in enumerate(pdf.pages,1):
            info={'Archivo':archivo,'Página':i,**extraer_info(pag.extract_text() or "")}
            tabla=pag.extract_table()
            if tabla and len(tabla)>1:
                enc=tabla[0]
                for fila in tabla[1:]:
                    filas.append({**info, **dict(zip(enc,fila))} if len(fila)==len(enc) else info)
            else: filas.append(info)

# --- exportar ---
df=pd.DataFrame(filas).replace('',pd.NA)
info_cols=['Archivo','Página','Invoice','Client','Date','Total invoice amount']
df=df.dropna(subset=[c for c in df.columns if c not in info_cols], how='all')
df.to_excel(destino,index=False)
print(f"✅ Excel generado: {destino}")

<p align="left">
  <img src="images/pdf_output.png" alt="Z-score" width="600"><br>
  <em>Así se ve la información en Excel que puedes obtener, sin trabajo manual, sin riesgo de errores</em>
</p>

### Conclusión

El código se puede optimizar bastante, pero aún así, pienso que es más eficiente y entendible que usar la funcionalidad incluída en ACL.